In [1]:
# =========================================================
# Cell 1 â€” Stable Low-VRAM Model Loading
# =========================================================

!pip -q install transformers accelerate bitsandbytes sentencepiece

import gc
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

# ---------------------------------------------------------
# CLEAN MEMORY
# ---------------------------------------------------------

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ---------------------------------------------------------
# MODEL CONFIGURATION
# ---------------------------------------------------------

MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# ---------------------------------------------------------
# TOKENIZER
# ---------------------------------------------------------

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Tokenizer loaded successfully.")

# ---------------------------------------------------------
# MODEL
# ---------------------------------------------------------

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

model.eval()

print("Model loaded successfully.")

# ---------------------------------------------------------
# GENERATION DEFAULTS
# ---------------------------------------------------------

GENERATION_CONFIG = {
    "max_new_tokens": 160,
    "do_sample": False,
    "repetition_penalty": 1.05,
}

# ---------------------------------------------------------
# GPU INFO
# ---------------------------------------------------------

if torch.cuda.is_available():
    print("\nGPU:", torch.cuda.get_device_name(0))

    allocated = round(
        torch.cuda.memory_allocated() / 1024**3,
        2
    )

    reserved = round(
        torch.cuda.memory_reserved() / 1024**3,
        2
    )

    print(f"VRAM Allocated: {allocated} GB")
    print(f"VRAM Reserved: {reserved} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully.

GPU: Tesla T4
VRAM Allocated: 2.06 GB
VRAM Reserved: 4.93 GB


In [2]:
# =========================================================
# Cell 2 â€” AST Extraction Utilities
# =========================================================

import ast

from typing import Dict
from typing import List
from typing import Optional
from typing import Any


# ---------------------------------------------------------
# SAFE AST UNPARSE
# ---------------------------------------------------------

def safe_unparse(node: Optional[ast.AST]) -> Optional[str]:
    """
    Safely converts an AST node back into source code.
    """

    if node is None:
        return None

    try:
        return ast.unparse(node)

    except Exception:
        return None


# ---------------------------------------------------------
# EXTRACT SOURCE SEGMENT
# ---------------------------------------------------------

def get_source_segment(code: str, node: ast.AST) -> str:
    """
    Safely extracts source code from an AST node.
    """

    return ast.get_source_segment(code, node) or ""


# ---------------------------------------------------------
# EXTRACT FUNCTION ARGUMENTS
# ---------------------------------------------------------

def extract_arguments(node: ast.FunctionDef) -> List[Dict[str, Any]]:
    """
    Extracts function arguments, annotations,
    and default values.
    """

    arguments = []

    positional_args = (
        node.args.posonlyargs
        + node.args.args
    )

    defaults = node.args.defaults

    padded_defaults = [None] * (
        len(positional_args) - len(defaults)
    ) + defaults

    for arg, default in zip(
        positional_args,
        padded_defaults,
    ):

        arguments.append({
            "name": arg.arg,
            "type": safe_unparse(arg.annotation),
            "default": safe_unparse(default),
        })

    if node.args.vararg:

        arguments.append({
            "name": "*" + node.args.vararg.arg,
            "type": safe_unparse(
                node.args.vararg.annotation
            ),
            "default": None,
        })

    for arg, default in zip(
        node.args.kwonlyargs,
        node.args.kw_defaults,
    ):

        arguments.append({
            "name": arg.arg,
            "type": safe_unparse(arg.annotation),
            "default": safe_unparse(default),
        })

    if node.args.kwarg:

        arguments.append({
            "name": "**" + node.args.kwarg.arg,
            "type": safe_unparse(
                node.args.kwarg.annotation
            ),
            "default": None,
        })

    return arguments


# ---------------------------------------------------------
# EXTRACT RETURN TYPE
# ---------------------------------------------------------

def extract_return_type(
    node: ast.FunctionDef
) -> Optional[str]:
    """
    Extracts function return type annotation.
    """

    return safe_unparse(node.returns)


# ---------------------------------------------------------
# CHECK IF FUNCTION HAS RETURN VALUE
# ---------------------------------------------------------

def function_has_return_value(
    node: ast.FunctionDef
) -> bool:
    """
    Determines whether the function returns
    a non-None value.
    """

    for child in ast.walk(node):

        if isinstance(child, ast.Return):

            if child.value is None:
                continue

            if (
                isinstance(child.value, ast.Constant)
                and child.value.value is None
            ):
                continue

            return True

    return False


# ---------------------------------------------------------
# MAIN FUNCTION INFO EXTRACTION
# ---------------------------------------------------------

def extract_function_info(code: str) -> Dict[str, Any]:
    """
    Parses Python code and extracts structured
    metadata from the first function definition found.
    """

    tree = ast.parse(code)

    for node in tree.body:

        if isinstance(node, ast.FunctionDef):

            function_code = get_source_segment(
                code,
                node
            )

            return {
                "function_name": node.name,
                "arguments": extract_arguments(node),
                "return_type": extract_return_type(node),
                "has_return_value": function_has_return_value(node),
                "function_code": function_code,
                "has_docstring": (
                    ast.get_docstring(node) is not None
                ),
            }

    raise ValueError(
        "No function definition found."
    )


# =========================================================
# TEST
# =========================================================

sample_code = '''
def add(a: int, b: int = 0) -> int:
    return a + b
'''

info = extract_function_info(sample_code)

print(info)

{'function_name': 'add', 'arguments': [{'name': 'a', 'type': 'int', 'default': None}, {'name': 'b', 'type': 'int', 'default': '0'}], 'return_type': 'int', 'has_return_value': True, 'function_code': 'def add(a: int, b: int = 0) -> int:\n    return a + b', 'has_docstring': False}


In [3]:
# =========================================================
# Cell 3 â€” Prompt Builder
# =========================================================


# ---------------------------------------------------------
# BUILD PROMPT
# ---------------------------------------------------------

def build_prompt(function_info: dict) -> str:
    """
    Builds a deterministic prompt for generating
    canonical Google-style docstrings.
    """

    function_code = function_info["function_code"]

    has_arguments = len(function_info["arguments"]) > 0
    has_return_value = function_info["has_return_value"]

    required_sections = []

    if has_arguments:
        required_sections.append("Args:")

    if has_return_value:
        required_sections.append("Returns:")

    sections_text = "\n".join(
        f"- {section}"
        for section in required_sections
    )

    prompt = f"""You are an expert Python software engineer.

Generate a canonical Google-style Python docstring.

Requirements:
- Return only the docstring
- Use triple double quotes
- Include Args: if arguments exist
- Include Returns: if the function returns a value
- Use 4-space indentation under sections
- Do not invent arguments

Function:

REQUIRED SECTIONS:
{sections_text}

CANONICAL FORMAT:
\"\"\"
Short description.

Args:
    x (int): Description.

Returns:
    int: Description.
\"\"\"

FUNCTION:
{function_code}
"""

    return prompt.strip()


# =========================================================
# TEST
# =========================================================

prompt = build_prompt(info)

print(prompt)

You are an expert Python software engineer.

Generate a canonical Google-style Python docstring.

Requirements:
- Return only the docstring
- Use triple double quotes
- Include Args: if arguments exist
- Include Returns: if the function returns a value
- Use 4-space indentation under sections
- Do not invent arguments

Function:

REQUIRED SECTIONS:
- Args:
- Returns:

CANONICAL FORMAT:
"""
Short description.

Args:
    x (int): Description.

Returns:
    int: Description.
"""

FUNCTION:
def add(a: int, b: int = 0) -> int:
    return a + b


In [4]:
# =========================================================
# Cell 4 â€” Generation Function
# =========================================================

import torch

# ---------------------------------------------------------
# EXTRACT GENERATED DOCSTRING
# ---------------------------------------------------------

def extract_generated_docstring(
    generated_text: str
) -> str:
    """
    Extracts the first complete triple-quoted
    docstring block from model output.
    """

    start_index = generated_text.find('"""')

    if start_index == -1:
        return generated_text.strip()

    end_index = generated_text.find(
        '"""',
        start_index + 3
    )

    if end_index == -1:
        return generated_text[start_index:].strip()

    return generated_text[
        start_index:end_index + 3
    ].strip()


# ---------------------------------------------------------
# GENERATION FUNCTION
# ---------------------------------------------------------

def generate_docstring(prompt: str) -> str:
    """
    Generates a raw Google-style docstring
    using chat-template inference.
    """

    messages = [
        {
            "role": "system",
            "content": (
                "You generate canonical Python "
                "Google-style docstrings."
            ),
        },
        {
            "role": "user",
            "content": prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            **GENERATION_CONFIG,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    raw_output = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )

    return raw_output.strip()


# =========================================================
# TEST
# =========================================================

generated_docstring = generate_docstring(prompt)

print(generated_docstring)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


```python
"""
Add two numbers together.

Args:
    a (int): The first number to add.
    b (int, optional): The second number to add. Defaults to 0.

Returns:
    int: The sum of the two numbers.
"""
```


In [5]:
# =========================================================
# Cell 5 â€” Cleaner / Parser / Canonical Renderer
# =========================================================

import re

from typing import Dict
from typing import List
from typing import Optional


# ---------------------------------------------------------
# DOCSTRING EXTRACTION
# ---------------------------------------------------------

def extract_docstring(
    raw_output: str
) -> str:
    """
    Extracts the first triple-quoted
    docstring block from model output.
    """

    match = re.search(
        r'""".*?"""',
        raw_output,
        re.DOTALL,
    )

    if not match:
        raise ValueError(
            "No valid docstring found."
        )

    return match.group(0)


# ---------------------------------------------------------
# REMOVE MARKDOWN FENCES
# ---------------------------------------------------------

def remove_markdown_fences(
    text: str
) -> str:
    """
    Removes markdown code fences from
    model output.
    """

    text = re.sub(
        r"```python",
        "",
        text,
    )

    text = re.sub(
        r"```",
        "",
        text,
    )

    return text.strip()


# ---------------------------------------------------------
# NORMALIZE WHITESPACE
# ---------------------------------------------------------

def normalize_whitespace(
    text: str
) -> str:
    """
    Performs minimal whitespace cleanup
    without restructuring semantics.
    """

    lines = [
        line.rstrip()
        for line in text.splitlines()
    ]

    normalized = []

    previous_blank = False

    for line in lines:

        if not line.strip():

            if not previous_blank:
                normalized.append("")

            previous_blank = True
            continue

        previous_blank = False

        normalized.append(line)

    return "\n".join(normalized).strip()


# ---------------------------------------------------------
# CLEAN DOCSTRING
# ---------------------------------------------------------

def clean_docstring(
    raw_output: str
) -> str:
    """
    Cleans raw model output while preserving
    semantic structure.
    """

    cleaned = remove_markdown_fences(
        raw_output
    )

    extracted = extract_docstring(
        cleaned
    )

    return normalize_whitespace(
        extracted
    )


# ---------------------------------------------------------
# PARSE ARGUMENT LINE
# ---------------------------------------------------------

def parse_argument_line(
    line: str
) -> Optional[Dict]:
    """
    Parses a Google-style argument line.
    """

    match = re.match(
        (
            r"^\s*"
            r"(\*{0,2}[a-zA-Z_][a-zA-Z0-9_]*)"
            r"\s*"
            r"\((.*?)\)"
            r":\s*(.+)$"
        ),
        line,
    )

    if not match:
        return None

    return {
        "name": match.group(1).strip(),
        "type": match.group(2).strip(),
        "description": (
            match.group(3).strip()
        ),
    }


# ---------------------------------------------------------
# PARSE RETURN LINE
# ---------------------------------------------------------

def parse_return_line(
    line: str
) -> Optional[Dict]:
    """
    Parses a Google-style return line.
    """

    match = re.match(
        r"^\s*(.+?):\s*(.+)$",
        line,
    )

    if not match:
        return None

    return {
        "type": match.group(1).strip(),
        "description": (
            match.group(2).strip()
        ),
    }


# ---------------------------------------------------------
# PARSE GOOGLE DOCSTRING
# ---------------------------------------------------------

def parse_google_docstring(
    docstring: str
) -> Dict:
    """
    Parses a Google-style docstring into
    a structured internal representation.
    """

    content = docstring.strip()

    content = content.removeprefix(
        '"""'
    )

    content = content.removesuffix(
        '"""'
    )

    lines = [
        line.rstrip()
        for line in content.splitlines()
    ]

    summary_lines = []

    parsed_args = []

    parsed_returns = None

    current_section = "summary"

    for raw_line in lines:

        line = raw_line.strip()

        if not line:
            continue

        if line == "Args:":
            current_section = "args"
            continue

        if line == "Returns:":
            current_section = "returns"
            continue

        if current_section == "summary":

            summary_lines.append(line)

        elif current_section == "args":

            parsed = parse_argument_line(
                line
            )

            if parsed:
                parsed_args.append(parsed)

        elif current_section == "returns":

            parsed = parse_return_line(
                line
            )

            if parsed:
                parsed_returns = parsed

    return {
        "summary": " ".join(
            summary_lines
        ),
        "args": parsed_args,
        "returns": parsed_returns,
    }


# ---------------------------------------------------------
# CANONICAL GOOGLE RENDERER
# ---------------------------------------------------------

def render_google_docstring(
    parsed_docstring: Dict
) -> str:
    """
    Renders a deterministic canonical
    Google-style docstring.
    """

    lines = ['"""']

    # Summary
    summary = (
        parsed_docstring["summary"]
        .strip()
    )

    lines.append(summary)

    # Args
    args = parsed_docstring["args"]

    if args:

        lines.append("")
        lines.append("Args:")

        for arg in args:

            lines.append(
                (
                    f"    {arg['name']} "
                    f"({arg['type']}): "
                    f"{arg['description']}"
                )
            )

    # Returns
    returns = parsed_docstring["returns"]

    if returns:

        lines.append("")
        lines.append("Returns:")

        lines.append(
            (
                f"    {returns['type']}: "
                f"{returns['description']}"
            )
        )

    lines.append('"""')

    return "\n".join(lines)


# =========================================================
# TEST
# =========================================================

raw_output = generate_docstring(
    prompt
)

cleaned_docstring = clean_docstring(
    raw_output
)

parsed_docstring = parse_google_docstring(
    cleaned_docstring
)

canonical_docstring = render_google_docstring(
    parsed_docstring
)

print(canonical_docstring)

print("\nParsed Structure:\n")

print(parsed_docstring)

"""
Add two numbers together.

Args:
    a (int): The first number to add.
    b (int, optional): The second number to add. Defaults to 0.

Returns:
    int: The sum of the two numbers.
"""

Parsed Structure:

{'summary': 'Add two numbers together.', 'args': [{'name': 'a', 'type': 'int', 'description': 'The first number to add.'}, {'name': 'b', 'type': 'int, optional', 'description': 'The second number to add. Defaults to 0.'}], 'returns': {'type': 'int', 'description': 'The sum of the two numbers.'}}


In [6]:
# =========================================================
# Cell 6 â€” Deterministic Docstring Insertion
# =========================================================

import ast
import re


# ---------------------------------------------------------
# INDENTATION DETECTION
# ---------------------------------------------------------

def detect_body_indentation(
    lines: list[str]
) -> str:
    """
    Detects indentation used inside
    the function body.
    """

    header_finished = False

    for line in lines:

        if not header_finished:

            if line.strip().endswith(":"):
                header_finished = True

            continue

        if not line.strip():
            continue

        match = re.match(
            r"^(\s+)",
            line,
        )

        if match:
            return match.group(1)

    return " " * 4


# ---------------------------------------------------------
# CHECK EXISTING DOCSTRING
# ---------------------------------------------------------

def function_has_docstring(
    function_code: str
) -> bool:
    """
    Detects whether a function already
    contains a docstring.
    """

    try:

        tree = ast.parse(
            function_code
        )

        node = tree.body[0]

        return (
            ast.get_docstring(node)
            is not None
        )

    except Exception:

        return False


# ---------------------------------------------------------
# INDENT DOCSTRING
# ---------------------------------------------------------

def indent_docstring(
    docstring: str,
    indentation: str,
) -> str:
    """
    Applies deterministic indentation
    to docstring lines.
    """

    indented_lines = []

    for line in docstring.splitlines():

        if line.strip():

            indented_lines.append(
                f"{indentation}{line}"
            )

        else:

            indented_lines.append("")

    return "\n".join(
        indented_lines
    )


# =========================================================
# STYLE-AGNOSTIC DOCSTRING INSERTION
# =========================================================

def insert_docstring(
    function_code: str,
    docstring: str,
) -> str:
    """
    Inserts a rendered docstring into
    function source code.
    """

    # -----------------------------------------------------
    # PRE-INSERTION VALIDATION
    # -----------------------------------------------------

    if function_has_docstring(
        function_code
    ):

        raise ValueError(
            "Function already contains "
            "a docstring."
        )

    lines = function_code.splitlines()

    if not lines:

        raise ValueError(
            "Empty function code."
        )

    # -----------------------------------------------------
    # DETECT BODY INDENTATION
    # -----------------------------------------------------

    body_indentation = (
        detect_body_indentation(
            lines
        )
    )

    # -----------------------------------------------------
    # INDENT DOCSTRING
    # -----------------------------------------------------

    indented_docstring = (
        indent_docstring(
            docstring,
            body_indentation,
        )
    )

    # -----------------------------------------------------
    # FIND INSERTION POINT
    # -----------------------------------------------------

    insert_index = 1

    while (
        insert_index < len(lines)
        and not lines[
            insert_index - 1
        ].strip().endswith(":")
    ):

        insert_index += 1

    if (
        insert_index == len(lines)
        and not lines[
            insert_index - 1
        ].strip().endswith(":")
    ):

        raise ValueError(
            (
                "Could not find a standalone "
                "function header ending with ':'. "
                "One-line function bodies are not "
                "supported by this insertion routine."
            )
        )

    # -----------------------------------------------------
    # INSERT DOCSTRING
    # -----------------------------------------------------

    new_lines = (
        lines[:insert_index]
        + [indented_docstring]
        + lines[insert_index:]
    )

    final_code = "\n".join(
        new_lines
    )

    # -----------------------------------------------------
    # FINAL AST VALIDATION
    # -----------------------------------------------------

    try:

        ast.parse(final_code)

    except SyntaxError as error:

        raise ValueError(
            "Generated code is invalid:\n"
            f"{error}"
        )

    return final_code


# =========================================================
# TEST
# =========================================================

rendered_docstring = canonical_docstring

final_function = insert_docstring(
    function_code=(
        info["function_code"]
    ),
    docstring=(
        rendered_docstring
    ),
)

print(final_function)

def add(a: int, b: int = 0) -> int:
    """
    Add two numbers together.

    Args:
        a (int): The first number to add.
        b (int, optional): The second number to add. Defaults to 0.

    Returns:
        int: The sum of the two numbers.
    """
    return a + b


In [7]:
# =========================================================
# Cell 6.5 â€” Structured Docstring Extraction
# =========================================================

from typing import Dict
from typing import List
from typing import Optional


# ---------------------------------------------------------
# EXTRACT DOCUMENTED ARGUMENTS
# ---------------------------------------------------------

def extract_documented_args(
    parsed_docstring: Dict
) -> List[str]:
    """
    Extracts documented argument names from
    the parsed internal representation.
    """

    documented_args = []

    for argument in parsed_docstring.get(
        "args",
        [],
    ):

        argument_name = argument.get(
            "name"
        )

        if argument_name:

            documented_args.append(
                argument_name
            )

    return documented_args


# ---------------------------------------------------------
# EXTRACT DOCUMENTED RETURN
# ---------------------------------------------------------

def extract_documented_return(
    parsed_docstring: Dict
) -> Optional[Dict]:
    """
    Extracts documented return metadata from
    the parsed internal representation.
    """

    returns = parsed_docstring.get(
        "returns"
    )

    if not returns:
        return None

    return {
        "type": returns.get("type"),
        "description": (
            returns.get(
                "description"
            )
        ),
    }


# ---------------------------------------------------------
# EXTRACT STRUCTURED DOCSTRING DATA
# ---------------------------------------------------------

def extract_docstring_structure(
    parsed_docstring: Dict
) -> Dict:
    """
    Extracts structured metadata from the
    parsed internal docstring representation.
    """

    return {
        "documented_args": (
            extract_documented_args(
                parsed_docstring
            )
        ),
        "documented_return": (
            extract_documented_return(
                parsed_docstring
            )
        ),
    }


# =========================================================
# TEST
# =========================================================

cleaned_docstring = clean_docstring(
    generated_docstring
)

parsed_docstring = parse_google_docstring(
    cleaned_docstring
)

docstring_structure = (
    extract_docstring_structure(
        parsed_docstring
    )
)

print(docstring_structure)

{'documented_args': ['a', 'b'], 'documented_return': {'type': 'int', 'description': 'The sum of the two numbers.'}}


In [8]:
# =========================================================
# Cell 7 â€” Universal Semantic Validation
# =========================================================

from typing import Dict
from typing import List


# ---------------------------------------------------------
# VALIDATE REQUIRED SECTIONS
# ---------------------------------------------------------

def validate_required_sections(
    function_info: Dict,
    schema: Dict,
) -> List[str]:
    """
    Validates required canonical sections.
    """

    errors = []

    if (
        function_info["arguments"]
        and not schema["args"]
    ):

        errors.append(
            "Missing required Args section."
        )

    if (
        function_info["has_return_value"]
        and not schema["returns"]
    ):

        errors.append(
            "Missing required Returns section."
        )

    return errors


# ---------------------------------------------------------
# VALIDATE SUMMARY
# ---------------------------------------------------------

def validate_summary(
    schema: Dict
) -> List[str]:
    """
    Validates summary presence.
    """

    errors = []

    summary = (
        schema.get(
            "summary",
            ""
        ).strip()
    )

    if not summary:

        errors.append(
            "Missing docstring summary."
        )

    return errors


# ---------------------------------------------------------
# VALIDATE ARGUMENT COVERAGE
# ---------------------------------------------------------

def validate_argument_coverage(
    function_info: Dict,
    schema: Dict,
) -> Dict:
    """
    Validates argument coverage consistency.
    """

    expected_args = sorted([
        argument["name"]
        for argument
        in function_info["arguments"]
    ])

    documented_args = sorted([
        argument["name"]
        for argument
        in schema.get(
            "args",
            [],
        )
    ])

    missing_args = [
        arg
        for arg
        in expected_args
        if arg
        not in documented_args
    ]

    hallucinated_args = [
        arg
        for arg
        in documented_args
        if arg
        not in expected_args
    ]

    return {
        "expected_args": (
            expected_args
        ),
        "documented_args": (
            documented_args
        ),
        "missing_args": (
            missing_args
        ),
        "hallucinated_args": (
            hallucinated_args
        ),
    }


# ---------------------------------------------------------
# VALIDATE DUPLICATE ARGUMENTS
# ---------------------------------------------------------

def validate_duplicate_arguments(
    schema: Dict
) -> List[str]:
    """
    Detects duplicate documented arguments.
    """

    errors = []

    arg_names = [
        arg["name"]
        for arg
        in schema.get(
            "args",
            []
        )
    ]

    duplicates = set([
        name
        for name
        in arg_names
        if arg_names.count(name) > 1
    ])

    for duplicate in duplicates:

        errors.append(
            f"Duplicate argument: {duplicate}"
        )

    return errors


# ---------------------------------------------------------
# VALIDATE ARGUMENT DESCRIPTIONS
# ---------------------------------------------------------

def validate_argument_descriptions(
    schema: Dict
) -> List[str]:
    """
    Validates argument descriptions.
    """

    errors = []

    for arg in schema.get(
        "args",
        []
    ):

        if not arg["description"].strip():

            errors.append(
                (
                    "Missing description for "
                    f'argument "{arg["name"]}".'
                )
            )

    return errors


# ---------------------------------------------------------
# VALIDATE RETURN DOCUMENTATION
# ---------------------------------------------------------

def validate_return_documentation(
    function_info: Dict,
    schema: Dict,
) -> List[str]:
    """
    Validates return documentation.
    """

    errors = []

    has_return_value = (
        function_info[
            "has_return_value"
        ]
    )

    documented_return = (
        schema.get(
            "returns"
        )
    )

    if (
        has_return_value
        and not documented_return
    ):

        errors.append(
            "Missing return documentation."
        )

    if (
        not has_return_value
        and documented_return
    ):

        errors.append(
            (
                "Unexpected return "
                "documentation."
            )
        )

    return errors


# ---------------------------------------------------------
# VALIDATE RETURN DESCRIPTION
# ---------------------------------------------------------

def validate_return_description(
    schema: Dict
) -> List[str]:
    """
    Validates return description.
    """

    errors = []

    returns = schema.get(
        "returns"
    )

    if returns:

        if not returns[
            "description"
        ].strip():

            errors.append(
                "Missing return description."
            )

    return errors


# ---------------------------------------------------------
# VALIDATE SCHEMA STRUCTURE
# ---------------------------------------------------------

def validate_schema_structure(
    schema: Dict
) -> List[str]:
    """
    Validates canonical schema structure.
    """

    errors = []

    required_keys = [
        "summary",
        "args",
        "returns",
        "raises",
        "examples",
        "notes",
        "yields",
    ]

    for key in required_keys:

        if key not in schema:

            errors.append(
                f"Missing schema key: {key}"
            )

    return errors


# ---------------------------------------------------------
# VALIDATE RAISES STRUCTURE
# ---------------------------------------------------------

def validate_raises_structure(
    schema: Dict
) -> List[str]:
    """
    Validates Raises entries.
    """

    errors = []

    for raise_item in schema.get(
        "raises",
        []
    ):

        if (
            not raise_item.get(
                "exception",
                ""
            ).strip()
        ):

            errors.append(
                "Empty exception name."
            )

        if (
            not raise_item.get(
                "description",
                ""
            ).strip()
        ):

            errors.append(
                "Empty raise description."
            )

    return errors


# ---------------------------------------------------------
# VALIDATE EXAMPLES STRUCTURE
# ---------------------------------------------------------

def validate_examples_structure(
    schema: Dict
) -> List[str]:
    """
    Validates Examples entries.
    """

    errors = []

    for example in schema.get(
        "examples",
        []
    ):

        if not example.strip():

            errors.append(
                "Empty example entry."
            )

    return errors


# ---------------------------------------------------------
# VALIDATE NOTES STRUCTURE
# ---------------------------------------------------------

def validate_notes_structure(
    schema: Dict
) -> List[str]:
    """
    Validates Notes entries.
    """

    errors = []

    for note in schema.get(
        "notes",
        []
    ):

        if not note.strip():

            errors.append(
                "Empty note entry."
            )

    return errors


# ---------------------------------------------------------
# MAIN SEMANTIC VALIDATION PIPELINE
# ---------------------------------------------------------

def validate_schema_semantics(
    function_info: Dict,
    schema: Dict,
) -> Dict:
    """
    Runs universal semantic validation
    against canonical schema.
    """

    validation_errors = []

    validation_errors.extend(
        validate_required_sections(
            function_info,
            schema,
        )
    )

    validation_errors.extend(
        validate_summary(
            schema
        )
    )

    validation_errors.extend(
        validate_duplicate_arguments(
            schema
        )
    )

    validation_errors.extend(
        validate_argument_descriptions(
            schema
        )
    )

    validation_errors.extend(
        validate_return_documentation(
            function_info,
            schema,
        )
    )

    validation_errors.extend(
        validate_return_description(
            schema
        )
    )

    validation_errors.extend(
        validate_schema_structure(
            schema
        )
    )

    validation_errors.extend(
        validate_raises_structure(
            schema
        )
    )

    validation_errors.extend(
        validate_examples_structure(
            schema
        )
    )

    validation_errors.extend(
        validate_notes_structure(
            schema
        )
    )

    argument_validation = (
        validate_argument_coverage(
            function_info,
            schema,
        )
    )

    if (
        argument_validation[
            "missing_args"
        ]
    ):

        validation_errors.append(
            (
                "Missing args: "
                f"{argument_validation['missing_args']}"
            )
        )

    if (
        argument_validation[
            "hallucinated_args"
        ]
    ):

        validation_errors.append(
            (
                "Hallucinated args: "
                f"{argument_validation['hallucinated_args']}"
            )
        )

    return {
        "expected_args": (
            argument_validation[
                "expected_args"
            ]
        ),
        "documented_args": (
            argument_validation[
                "documented_args"
            ]
        ),
        "missing_args": (
            argument_validation[
                "missing_args"
            ]
        ),
        "hallucinated_args": (
            argument_validation[
                "hallucinated_args"
            ]
        ),
        "errors": (
            validation_errors
        ),
        "is_valid": (
            len(validation_errors)
            == 0
        ),
    }


# =========================================================
# TEST
# =========================================================

print(
    (
        "Semantic validation utilities loaded. "
        "The validation test runs after normalized_schema "
        "is created in Cell 8."
    )
)

Semantic validation utilities loaded. The validation test runs after normalized_schema is created in Cell 8.


In [9]:
# =========================================================
# Cell 7.5 â€” Google Render Validation
# =========================================================

import re

from typing import Dict
from typing import List


# ---------------------------------------------------------
# VALIDATE GOOGLE HEADERS
# ---------------------------------------------------------

def validate_google_headers(
    docstring: str
) -> List[str]:
    """
    Validates canonical Google headers.
    """

    errors = []

    forbidden_headers = [
        "Parameters:",
        "Arguments:",
        "Params:",
    ]

    for header in forbidden_headers:

        if header in docstring:

            errors.append(
                (
                    "Forbidden header detected: "
                    f"{header}"
                )
            )

    return errors


# ---------------------------------------------------------
# VALIDATE INDENTATION
# ---------------------------------------------------------

def validate_google_indentation(
    docstring: str
) -> List[str]:
    """
    Validates Google-style indentation.
    """

    errors = []

    lines = docstring.splitlines()

    inside_section = False

    for line in lines:

        stripped = line.strip()

        if stripped in {
            "Args:",
            "Returns:",
            "Raises:",
            "Examples:",
            "Notes:",
            "Yields:",
        }:

            inside_section = True
            continue

        if not stripped:
            continue

        if (
            inside_section
            and not line.startswith(
                "    "
            )
        ):

            inside_section = False

        if inside_section:

            if not re.match(
                r"^\s{4}\S",
                line,
            ):

                errors.append(
                    (
                        "Invalid indentation: "
                        f"{line}"
                    )
                )

    return errors


# ---------------------------------------------------------
# VALIDATE SECTION ORDER
# ---------------------------------------------------------

def validate_google_section_order(
    docstring: str
) -> List[str]:
    """
    Validates canonical Google section order.
    """

    errors = []

    canonical_order = [
        "Args:",
        "Returns:",
        "Raises:",
        "Yields:",
        "Examples:",
        "Notes:",
    ]

    found_sections = []

    for line in docstring.splitlines():

        stripped = line.strip()

        if stripped in canonical_order:

            found_sections.append(
                stripped
            )

    found_indexes = [
        canonical_order.index(section)
        for section
        in found_sections
    ]

    if found_indexes != sorted(found_indexes):

        errors.append(
            "Invalid Google section order."
        )

    return errors


# ---------------------------------------------------------
# VALIDATE DOCSTRING QUOTES
# ---------------------------------------------------------

def validate_docstring_quotes(
    docstring: str
) -> List[str]:
    """
    Validates triple quotes presence.
    """

    errors = []

    stripped = docstring.strip()

    if not stripped.startswith('"""'):

        errors.append(
            "Missing opening triple quotes."
        )

    if not stripped.endswith('"""'):

        errors.append(
            "Missing closing triple quotes."
        )

    return errors


# ---------------------------------------------------------
# MAIN GOOGLE VALIDATION PIPELINE
# ---------------------------------------------------------

def validate_google_render(
    rendered_docstring: str
) -> Dict:
    """
    Runs Google-style render validation.
    """

    validation_errors = []

    validation_errors.extend(
        validate_google_headers(
            rendered_docstring
        )
    )

    validation_errors.extend(
        validate_google_indentation(
            rendered_docstring
        )
    )

    validation_errors.extend(
        validate_google_section_order(
            rendered_docstring
        )
    )

    validation_errors.extend(
        validate_docstring_quotes(
            rendered_docstring
        )
    )

    return {
        "errors": (
            validation_errors
        ),
        "is_valid": (
            len(validation_errors)
            == 0
        ),
    }


# =========================================================
# TEST
# =========================================================

google_validation = (
    validate_google_render(
        rendered_docstring
    )
)

print(google_validation)

{'errors': [], 'is_valid': True}


In [10]:
# =========================================================
# Cell 8 â€” Google Style Parsing + Normalization + Rendering
# =========================================================

import re


# =========================================================
# SECTION HEADERS
# =========================================================

GOOGLE_SECTIONS = {
    "Args:": "args",
    "Returns:": "returns",
    "Raises:": "raises",
    "Examples:": "examples",
    "Notes:": "notes",
    "Yields:": "yields",
}


# =========================================================
# PARSE GOOGLE DOCSTRING
# =========================================================

def parse_google_docstring(docstring: str) -> dict:
    """
    Parses a Google-style docstring into structured sections.
    """

    parsed = {
        "summary": "",
        "args": [],
        "returns": None,
        "raises": [],
        "examples": [],
        "notes": [],
        "yields": None,
    }

    lines = [
        line.rstrip()
        for line
        in docstring.strip().splitlines()
    ]

    current_section = "summary"

    summary_lines = []

    for line in lines:

        stripped = line.strip()

        # Skip docstring quotes
        if stripped in ['"""', "'''"]:
            continue

        # Detect section headers
        if stripped in GOOGLE_SECTIONS:
            current_section = GOOGLE_SECTIONS[
                stripped
            ]
            continue

        # Skip empty lines
        if not stripped:
            continue

        # =================================================
        # SUMMARY
        # =================================================

        if current_section == "summary":
            summary_lines.append(stripped)

        # =================================================
        # ARGS
        # =================================================

        elif current_section == "args":

            match = re.match(
                (
                    r"(\*{0,2}\w+)"
                    r"\s*"
                    r"\((.*?)\)"
                    r":\s*(.*)"
                ),
                stripped,
            )

            if match:
                name, arg_type, description = (
                    match.groups()
                )

                parsed["args"].append({
                    "name": name.strip(),
                    "type": arg_type.strip(),
                    "description": description.strip(),
                })

        # =================================================
        # RETURNS
        # =================================================

        elif current_section == "returns":

            match = re.match(
                r"(.*?)\s*:\s*(.*)",
                stripped,
            )

            if match:
                return_type, description = (
                    match.groups()
                )

                parsed["returns"] = {
                    "type": return_type.strip(),
                    "description": description.strip(),
                }

        # =================================================
        # RAISES
        # =================================================

        elif current_section == "raises":

            match = re.match(
                r"([\w.]+)\s*:\s*(.*)",
                stripped,
            )

            if match:
                exception, description = (
                    match.groups()
                )

                parsed["raises"].append({
                    "exception": exception.strip(),
                    "description": description.strip(),
                })

        # =================================================
        # EXAMPLES
        # =================================================

        elif current_section == "examples":
            parsed["examples"].append(stripped)

        # =================================================
        # NOTES
        # =================================================

        elif current_section == "notes":
            parsed["notes"].append(stripped)

        # =================================================
        # YIELDS
        # =================================================

        elif current_section == "yields":

            match = re.match(
                r"(.*?)\s*:\s*(.*)",
                stripped,
            )

            if match:
                yield_type, description = (
                    match.groups()
                )

                parsed["yields"] = {
                    "type": yield_type.strip(),
                    "description": description.strip(),
                }

    parsed["summary"] = " ".join(
        summary_lines
    ).strip()

    return parsed


# =========================================================
# NORMALIZATION
# =========================================================

def normalize_docstring_data(
    parsed_docstring: dict,
    function_info: dict
) -> dict:
    """
    Normalizes parsed docstring data into a canonical schema.
    Uses function annotations first, then falls back to parsed types.
    """

    normalized_args = []

    function_args = {
        arg["name"]: arg
        for arg in function_info.get(
            "arguments",
            []
        )
    }

    for parsed_arg in parsed_docstring["args"]:

        arg_name = parsed_arg["name"]

        annotation_type = (
            function_args
            .get(arg_name, {})
            .get("type")
        )

        normalized_args.append({
            "name": arg_name,
            "type": annotation_type or parsed_arg["type"],
            "description": parsed_arg["description"],
        })

    normalized_returns = None

    if parsed_docstring["returns"]:

        normalized_returns = {
            "type": (
                function_info.get("return_type")
                or parsed_docstring["returns"]["type"]
            ),
            "description": parsed_docstring[
                "returns"
            ]["description"],
        }

    normalized_yields = None

    if parsed_docstring["yields"]:

        normalized_yields = {
            "type": parsed_docstring["yields"]["type"],
            "description": parsed_docstring[
                "yields"
            ]["description"],
        }

    normalized = {
        "summary": parsed_docstring["summary"],

        "args": normalized_args,

        "returns": normalized_returns,

        "raises": parsed_docstring["raises"],

        "examples": parsed_docstring["examples"],

        "notes": parsed_docstring["notes"],

        "yields": normalized_yields,
    }

    return normalized


# =========================================================
# GOOGLE STYLE RENDERER
# =========================================================

def render_google_docstring(schema: dict) -> str:
    """
    Renders a canonical schema into strict Google-style format.
    Automatically omits missing sections.
    """

    lines = ['"""']

    # =====================================================
    # SUMMARY
    # =====================================================

    if schema["summary"]:
        lines.append(
            schema["summary"].strip()
        )

    # =====================================================
    # ARGS
    # =====================================================

    if schema["args"]:

        lines.append("")
        lines.append("Args:")

        for arg in schema["args"]:

            arg_type = arg["type"] or "Any"

            lines.append(
                (
                    f'    {arg["name"]} '
                    f'({arg_type}): '
                    f'{arg["description"]}'
                )
            )

    # =====================================================
    # RETURNS
    # =====================================================

    if schema["returns"]:

        lines.append("")
        lines.append("Returns:")

        lines.append(
            (
                f'    {schema["returns"]["type"]}: '
                f'{schema["returns"]["description"]}'
            )
        )

    # =====================================================
    # RAISES
    # =====================================================

    if schema["raises"]:

        lines.append("")
        lines.append("Raises:")

        for item in schema["raises"]:

            lines.append(
                (
                    f'    {item["exception"]}: '
                    f'{item["description"]}'
                )
            )

    # =====================================================
    # YIELDS
    # =====================================================

    if schema["yields"]:

        lines.append("")
        lines.append("Yields:")

        lines.append(
            (
                f'    {schema["yields"]["type"]}: '
                f'{schema["yields"]["description"]}'
            )
        )

    # =====================================================
    # EXAMPLES
    # =====================================================

    if schema["examples"]:

        lines.append("")
        lines.append("Examples:")

        for example in schema["examples"]:
            lines.append(
                f"    {example}"
            )

    # =====================================================
    # NOTES
    # =====================================================

    if schema["notes"]:

        lines.append("")
        lines.append("Notes:")

        for note in schema["notes"]:
            lines.append(
                f"    {note}"
            )

    lines.append('"""')

    return "\n".join(lines)


# =========================================================
# COMPLETE PIPELINE HELPER
# =========================================================

def build_google_docstring(
    cleaned_docstring: str,
    function_info: dict
) -> dict:
    """
    Full Google-style processing pipeline.
    """

    parsed = parse_google_docstring(
        cleaned_docstring
    )

    normalized = normalize_docstring_data(
        parsed_docstring=parsed,
        function_info=function_info
    )

    rendered = render_google_docstring(
        normalized
    )

    return {
        "parsed": parsed,
        "normalized": normalized,
        "rendered": rendered,
    }


# =========================================================
# TEST â€” FULL GOOGLE PIPELINE
# =========================================================

cleaned_docstring = clean_docstring(
    generated_docstring
)

parsed_docstring = parse_google_docstring(
    cleaned_docstring
)

normalized_schema = normalize_docstring_data(
    parsed_docstring=parsed_docstring,
    function_info=info,
)

semantic_validation = validate_schema_semantics(
    function_info=info,
    schema=normalized_schema,
)

rendered_docstring = render_google_docstring(
    normalized_schema
)

print("==================================================")
print("PARSED")
print("==================================================")
print(parsed_docstring)

print("\n==================================================")
print("NORMALIZED")
print("==================================================")
print(normalized_schema)

print("\n==================================================")
print("SEMANTIC VALIDATION")
print("==================================================")
print(semantic_validation)

print("\n==================================================")
print("RENDERED")
print("==================================================")
print(rendered_docstring)

PARSED
{'summary': 'Add two numbers together.', 'args': [{'name': 'a', 'type': 'int', 'description': 'The first number to add.'}, {'name': 'b', 'type': 'int, optional', 'description': 'The second number to add. Defaults to 0.'}], 'returns': {'type': 'int', 'description': 'The sum of the two numbers.'}, 'raises': [], 'examples': [], 'notes': [], 'yields': None}

NORMALIZED
{'summary': 'Add two numbers together.', 'args': [{'name': 'a', 'type': 'int', 'description': 'The first number to add.'}, {'name': 'b', 'type': 'int', 'description': 'The second number to add. Defaults to 0.'}], 'returns': {'type': 'int', 'description': 'The sum of the two numbers.'}, 'raises': [], 'examples': [], 'notes': [], 'yields': None}

SEMANTIC VALIDATION
{'expected_args': ['a', 'b'], 'documented_args': ['a', 'b'], 'missing_args': [], 'hallucinated_args': [], 'errors': [], 'is_valid': True}

RENDERED
"""
Add two numbers together.

Args:
    a (int): The first number to add.
    b (int): The second number to a

In [11]:
# =========================================================
# Cell 8.5 â€” Minimal Google Render Validation
# =========================================================

import re

from typing import Dict
from typing import List


# ---------------------------------------------------------
# VALIDATE TRIPLE QUOTES
# ---------------------------------------------------------

def validate_google_quotes(
    docstring: str
) -> List[str]:
    """
    Validates triple quotes presence.
    """

    errors = []

    stripped = docstring.strip()

    if not stripped.startswith('"""'):

        errors.append(
            "Missing opening triple quotes."
        )

    if not stripped.endswith('"""'):

        errors.append(
            "Missing closing triple quotes."
        )

    return errors


# ---------------------------------------------------------
# VALIDATE GOOGLE INDENTATION
# ---------------------------------------------------------

def validate_google_indentation(
    docstring: str
) -> List[str]:
    """
    Validates section indentation.
    """

    errors = []

    section_headers = {
        "Args:",
        "Returns:",
        "Raises:",
        "Examples:",
        "Notes:",
        "Yields:",
    }

    lines = docstring.splitlines()

    inside_section = False

    for line in lines:

        stripped = line.strip()

        if stripped in section_headers:

            inside_section = True
            continue

        if not stripped:

            continue

        if (
            inside_section
            and not line.startswith(
                "    "
            )
        ):

            inside_section = False

        if inside_section:

            if not re.match(
                r"^\s{4}\S",
                line
            ):

                errors.append(
                    (
                        "Invalid indentation: "
                        f"{line}"
                    )
                )

    return errors


# ---------------------------------------------------------
# VALIDATE FORBIDDEN HEADERS
# ---------------------------------------------------------

def validate_google_headers(
    docstring: str
) -> List[str]:
    """
    Validates canonical Google headers.
    """

    errors = []

    forbidden_headers = [
        "Parameters:",
        "Arguments:",
        "Params:",
    ]

    for header in forbidden_headers:

        if header in docstring:

            errors.append(
                (
                    "Forbidden header detected: "
                    f"{header}"
                )
            )

    return errors


# ---------------------------------------------------------
# VALIDATE SECTION ORDER
# ---------------------------------------------------------

def validate_google_section_order(
    docstring: str
) -> List[str]:
    """
    Validates canonical Google section order.
    """

    errors = []

    canonical_order = [
        "Args:",
        "Returns:",
        "Raises:",
        "Yields:",
        "Examples:",
        "Notes:",
    ]

    found_sections = []

    for line in docstring.splitlines():

        stripped = line.strip()

        if stripped in canonical_order:

            found_sections.append(
                stripped
            )

    found_indexes = [
        canonical_order.index(section)
        for section
        in found_sections
    ]

    if found_indexes != sorted(found_indexes):

        errors.append(
            "Invalid Google section order."
        )

    return errors


# ---------------------------------------------------------
# MAIN GOOGLE VALIDATION
# ---------------------------------------------------------

def validate_google_render(
    rendered_docstring: str
) -> Dict:
    """
    Runs minimal Google-style validation.
    """

    validation_errors = []

    validation_errors.extend(
        validate_google_quotes(
            rendered_docstring
        )
    )

    validation_errors.extend(
        validate_google_indentation(
            rendered_docstring
        )
    )

    validation_errors.extend(
        validate_google_headers(
            rendered_docstring
        )
    )

    validation_errors.extend(
        validate_google_section_order(
            rendered_docstring
        )
    )

    return {
        "errors": (
            validation_errors
        ),
        "is_valid": (
            len(validation_errors)
            == 0
        ),
    }


# =========================================================
# TEST
# =========================================================

google_render_validation = (
    validate_google_render(
        rendered_docstring
    )
)

print(google_render_validation)

{'errors': [], 'is_valid': True}


In [12]:
# =========================================================
# Cell 9 â€” NumPy Style Renderer
# =========================================================

from typing import Dict


# ---------------------------------------------------------
# NUMPY SECTION HELPER
# ---------------------------------------------------------

def build_numpy_section_header(
    title: str
) -> list[str]:
    """
    Builds canonical NumPy section headers.
    """

    return [
        title,
        "-" * len(title),
    ]


# ---------------------------------------------------------
# NUMPY RENDERER
# ---------------------------------------------------------

def render_numpy_docstring(
    schema: Dict
) -> str:
    """
    Renders canonical schema into
    NumPy-style docstring format.
    """

    lines = ['"""']

    # =====================================================
    # SUMMARY
    # =====================================================

    if schema["summary"]:

        lines.append(
            schema["summary"].strip()
        )

    # =====================================================
    # PARAMETERS
    # =====================================================

    if schema["args"]:

        lines.append("")

        lines.extend(
            build_numpy_section_header(
                "Parameters"
            )
        )

        for arg in schema["args"]:

            arg_type = (
                arg["type"]
                or "Any"
            )

            lines.append(
                f'{arg["name"]} : {arg_type}'
            )

            lines.append(
                f'    {arg["description"]}'
            )

    # =====================================================
    # RETURNS
    # =====================================================

    if schema["returns"]:

        lines.append("")

        lines.extend(
            build_numpy_section_header(
                "Returns"
            )
        )

        lines.append(
            schema["returns"]["type"]
        )

        lines.append(
            (
                f'    '
                f'{schema["returns"]["description"]}'
            )
        )

    # =====================================================
    # RAISES
    # =====================================================

    if schema["raises"]:

        lines.append("")

        lines.extend(
            build_numpy_section_header(
                "Raises"
            )
        )

        for raise_item in schema["raises"]:

            lines.append(
                raise_item["exception"]
            )

            lines.append(
                (
                    f'    '
                    f'{raise_item["description"]}'
                )
            )

    # =====================================================
    # EXAMPLES
    # =====================================================

    if schema["examples"]:

        lines.append("")

        lines.extend(
            build_numpy_section_header(
                "Examples"
            )
        )

        for example in schema["examples"]:

            lines.append(
                example
            )

    # =====================================================
    # NOTES
    # =====================================================

    if schema["notes"]:

        lines.append("")

        lines.extend(
            build_numpy_section_header(
                "Notes"
            )
        )

        for note in schema["notes"]:

            lines.append(
                note
            )

    # =====================================================
    # YIELDS
    # =====================================================

    if schema["yields"]:

        lines.append("")

        lines.extend(
            build_numpy_section_header(
                "Yields"
            )
        )

        lines.append(
            schema["yields"]["type"]
        )

        lines.append(
            (
                f'    '
                f'{schema["yields"]["description"]}'
            )
        )

    lines.append('"""')

    return "\n".join(lines)


# =========================================================
# TEST
# =========================================================

numpy_rendered_docstring = (
    render_numpy_docstring(
        normalized_schema
    )
)

print(
    numpy_rendered_docstring
)

"""
Add two numbers together.

Parameters
----------
a : int
    The first number to add.
b : int
    The second number to add. Defaults to 0.

Returns
-------
int
    The sum of the two numbers.
"""


In [13]:
# =========================================================
# Cell 9.5 â€” Minimal NumPy Render Validation
# =========================================================

import re

from typing import Dict
from typing import List


# ---------------------------------------------------------
# VALIDATE TRIPLE QUOTES
# ---------------------------------------------------------

def validate_numpy_quotes(
    docstring: str
) -> List[str]:
    """
    Validates triple quotes presence.
    """

    errors = []

    stripped = docstring.strip()

    if not stripped.startswith('"""'):

        errors.append(
            "Missing opening triple quotes."
        )

    if not stripped.endswith('"""'):

        errors.append(
            "Missing closing triple quotes."
        )

    return errors


# ---------------------------------------------------------
# VALIDATE NUMPY SECTION UNDERLINES
# ---------------------------------------------------------

def validate_numpy_underlines(
    docstring: str
) -> List[str]:
    """
    Validates NumPy section underline formatting.
    """

    errors = []

    lines = docstring.splitlines()

    for index, line in enumerate(lines[:-1]):

        stripped = line.strip()

        # Skip empty lines
        if not stripped:
            continue

        # Detect possible section headers
        next_line = lines[index + 1].strip()

        if re.fullmatch(
            r"-{3,}",
            next_line
        ):

            if len(next_line) != len(stripped):

                errors.append(
                    (
                        "Invalid underline length "
                        f'for section "{stripped}".'
                    )
                )

    return errors


# ---------------------------------------------------------
# VALIDATE NUMPY INDENTATION
# ---------------------------------------------------------

def validate_numpy_indentation(
    docstring: str
) -> List[str]:
    """
    Validates NumPy indentation.
    """

    errors = []

    lines = docstring.splitlines()

    for index, line in enumerate(lines[:-1]):

        stripped = line.strip()

        # Parameter definition
        if re.match(
            r"^\w+\s*:\s*.+",
            stripped
        ):

            description_line = (
                lines[index + 1]
            )

            if not description_line.startswith(
                "    "
            ):

                errors.append(
                    (
                        "Invalid parameter "
                        f'indentation after "{stripped}".'
                    )
                )

        # Returns/Raises/Yields blocks
        elif (
            stripped
            and not stripped.startswith('"')
            and not stripped.startswith("-")
        ):

            if (
                index + 1 < len(lines)
            ):

                next_line = (
                    lines[index + 1]
                )

                if (
                    next_line.strip()
                    and not next_line.startswith(
                        "    "
                    )
                    and not re.fullmatch(
                        r"-{3,}",
                        next_line.strip()
                    )
                ):

                    continue

    return errors


# ---------------------------------------------------------
# VALIDATE NUMPY SECTION HEADERS
# ---------------------------------------------------------

def validate_numpy_headers(
    docstring: str
) -> List[str]:
    """
    Detects forbidden Google-style headers.
    """

    errors = []

    forbidden_headers = [
        "Args:",
        "Returns:",
        "Raises:",
    ]

    for header in forbidden_headers:

        if header in docstring:

            errors.append(
                (
                    "Forbidden Google header "
                    f"detected: {header}"
                )
            )

    return errors


# ---------------------------------------------------------
# MAIN NUMPY VALIDATION
# ---------------------------------------------------------

def validate_numpy_render(
    rendered_docstring: str
) -> Dict:
    """
    Runs minimal NumPy-style validation.
    """

    validation_errors = []

    validation_errors.extend(
        validate_numpy_quotes(
            rendered_docstring
        )
    )

    validation_errors.extend(
        validate_numpy_underlines(
            rendered_docstring
        )
    )

    validation_errors.extend(
        validate_numpy_indentation(
            rendered_docstring
        )
    )

    validation_errors.extend(
        validate_numpy_headers(
            rendered_docstring
        )
    )

    return {
        "errors": (
            validation_errors
        ),
        "is_valid": (
            len(validation_errors)
            == 0
        ),
    }


# =========================================================
# TEST
# =========================================================

numpy_render_validation = (
    validate_numpy_render(
        numpy_rendered_docstring
    )
)

print(
    numpy_render_validation
)

{'errors': [], 'is_valid': True}


In [14]:
test_schema = {
    "summary": "Processes input data.",

    "args": [
        {
            "name": "data",
            "type": "list[str]",
            "description": "Input data to process."
        }
    ],

    "returns": {
        "type": "dict",
        "description": "Processed results."
    },

    "raises": [
        {
            "exception": "ValueError",
            "description": "Raised when data is empty."
        }
    ],

    "examples": [
        ">>> process_data(['a', 'b'])",
        "{'count': 2}"
    ],

    "notes": [
        "This function is experimental."
    ],

    "yields": {
        "type": "str",
        "description": "Processed items."
    }
}

In [15]:
render_google_docstring(test_schema)

'"""\nProcesses input data.\n\nArgs:\n    data (list[str]): Input data to process.\n\nReturns:\n    dict: Processed results.\n\nRaises:\n    ValueError: Raised when data is empty.\n\nYields:\n    str: Processed items.\n\nExamples:\n    >>> process_data([\'a\', \'b\'])\n    {\'count\': 2}\n\nNotes:\n    This function is experimental.\n"""'

In [16]:
render_numpy_docstring(test_schema)

'"""\nProcesses input data.\n\nParameters\n----------\ndata : list[str]\n    Input data to process.\n\nReturns\n-------\ndict\n    Processed results.\n\nRaises\n------\nValueError\n    Raised when data is empty.\n\nExamples\n--------\n>>> process_data([\'a\', \'b\'])\n{\'count\': 2}\n\nNotes\n-----\nThis function is experimental.\n\nYields\n------\nstr\n    Processed items.\n"""'

In [17]:
# =========================================================
# Cell 10 â€” Sphinx Style Renderer
# =========================================================

from typing import Dict


# ---------------------------------------------------------
# SPHINX RENDERER
# ---------------------------------------------------------

def render_sphinx_docstring(
    schema: Dict
) -> str:
    """
    Renders canonical schema into
    Sphinx-style docstring format.
    """

    lines = ['"""']

    # =====================================================
    # SUMMARY
    # =====================================================

    if schema["summary"]:

        lines.append(
            schema["summary"].strip()
        )

    # =====================================================
    # PARAMETERS
    # =====================================================

    if schema["args"]:

        lines.append("")

        for arg in schema["args"]:

            arg_type = (
                arg["type"]
                or "Any"
            )

            lines.append(
                (
                    f':param {arg["name"]}: '
                    f'{arg["description"]}'
                )
            )

            lines.append(
                (
                    f':type {arg["name"]}: '
                    f'{arg_type}'
                )
            )

    # =====================================================
    # RETURNS
    # =====================================================

    if schema["returns"]:

        lines.append("")

        lines.append(
            (
                f':returns: '
                f'{schema["returns"]["description"]}'
            )
        )

        lines.append(
            (
                f':rtype: '
                f'{schema["returns"]["type"]}'
            )
        )

    # =====================================================
    # RAISES
    # =====================================================

    if schema["raises"]:

        lines.append("")

        for raise_item in schema["raises"]:

            lines.append(
                (
                    f':raises {raise_item["exception"]}: '
                    f'{raise_item["description"]}'
                )
            )

    # =====================================================
    # EXAMPLES
    # =====================================================

    if schema["examples"]:

        lines.append("")

        lines.append("Example:")

        for example in schema["examples"]:

            lines.append(
                example
            )

    # =====================================================
    # NOTES
    # =====================================================

    if schema["notes"]:

        lines.append("")

        lines.append("Note:")

        for note in schema["notes"]:

            lines.append(
                note
            )

    # =====================================================
    # YIELDS
    # =====================================================

    if schema["yields"]:

        lines.append("")

        lines.append(
            (
                f':yields: '
                f'{schema["yields"]["description"]}'
            )
        )

        lines.append(
            (
                f':ytype: '
                f'{schema["yields"]["type"]}'
            )
        )

    lines.append('"""')

    return "\n".join(lines)


# =========================================================
# TEST
# =========================================================

sphinx_rendered_docstring = (
    render_sphinx_docstring(
        test_schema
    )
)

print(
    sphinx_rendered_docstring
)

"""
Processes input data.

:param data: Input data to process.
:type data: list[str]

:returns: Processed results.
:rtype: dict

:raises ValueError: Raised when data is empty.

Example:
>>> process_data(['a', 'b'])
{'count': 2}

Note:
This function is experimental.

:yields: Processed items.
:ytype: str
"""


In [18]:
# =========================================================
# Cell 10.5 â€” Minimal Sphinx Render Validation
# =========================================================

import re

from typing import Dict
from typing import List


# ---------------------------------------------------------
# VALIDATE TRIPLE QUOTES
# ---------------------------------------------------------

def validate_sphinx_quotes(
    docstring: str
) -> List[str]:
    """
    Validates triple quotes presence.
    """

    errors = []

    stripped = docstring.strip()

    if not stripped.startswith('"""'):

        errors.append(
            "Missing opening triple quotes."
        )

    if not stripped.endswith('"""'):

        errors.append(
            "Missing closing triple quotes."
        )

    return errors


# ---------------------------------------------------------
# VALIDATE FORBIDDEN HEADERS
# ---------------------------------------------------------

def validate_sphinx_headers(
    docstring: str
) -> List[str]:
    """
    Detects forbidden Google and
    NumPy section headers.
    """

    errors = []

    forbidden_headers = [
        "Args:",
        "Returns:",
        "Raises:",
        "Parameters",
        "----------",
    ]

    for header in forbidden_headers:

        if header in docstring:

            errors.append(
                (
                    "Forbidden header detected: "
                    f"{header}"
                )
            )

    return errors


# ---------------------------------------------------------
# VALIDATE SPHINX DIRECTIVES
# ---------------------------------------------------------

def validate_sphinx_directives(
    docstring: str
) -> List[str]:
    """
    Validates Sphinx directive syntax.
    """

    errors = []

    valid_directives = [
        ":param",
        ":type",
        ":returns:",
        ":rtype:",
        ":raises",
        ":yields:",
        ":ytype:",
    ]

    lines = docstring.splitlines()

    for line in lines:

        stripped = line.strip()

        if stripped.startswith(":"):

            if not any(
                stripped.startswith(
                    directive
                )
                for directive
                in valid_directives
            ):

                errors.append(
                    (
                        "Invalid Sphinx directive: "
                        f"{stripped}"
                    )
                )

    return errors


# ---------------------------------------------------------
# VALIDATE BASIC INDENTATION
# ---------------------------------------------------------

def validate_sphinx_indentation(
    docstring: str
) -> List[str]:
    """
    Validates indentation consistency.
    """

    errors = []

    lines = docstring.splitlines()

    for line in lines:

        if not line.strip():
            continue

        leading_spaces = (
            len(line)
            - len(line.lstrip())
        )

        if (
            leading_spaces % 4 != 0
            and leading_spaces != 0
        ):

            errors.append(
                (
                    "Non-canonical indentation: "
                    f"{line}"
                )
            )

    return errors


# ---------------------------------------------------------
# MAIN SPHINX VALIDATION
# ---------------------------------------------------------

def validate_sphinx_render(
    rendered_docstring: str
) -> Dict:
    """
    Runs minimal Sphinx-style validation.
    """

    validation_errors = []

    validation_errors.extend(
        validate_sphinx_quotes(
            rendered_docstring
        )
    )

    validation_errors.extend(
        validate_sphinx_headers(
            rendered_docstring
        )
    )

    validation_errors.extend(
        validate_sphinx_directives(
            rendered_docstring
        )
    )

    validation_errors.extend(
        validate_sphinx_indentation(
            rendered_docstring
        )
    )

    return {
        "errors": (
            validation_errors
        ),
        "is_valid": (
            len(validation_errors)
            == 0
        ),
    }


# =========================================================
# TEST
# =========================================================

sphinx_validation_result = (
    validate_sphinx_render(
        sphinx_rendered_docstring
    )
)

print(
    sphinx_validation_result
)

{'errors': [], 'is_valid': True}


In [19]:
# =========================================================
# Cell 11 â€” Universal Pipeline Runner
# =========================================================

import ast

from typing import Callable
from typing import Dict


# ---------------------------------------------------------
# STYLE REGISTRY
# ---------------------------------------------------------

RENDERERS = {
    "google": render_google_docstring,
    "numpy": render_numpy_docstring,
    "sphinx": render_sphinx_docstring,
}

STYLE_VALIDATORS = {
    "google": validate_google_render,
    "numpy": validate_numpy_render,
    "sphinx": validate_sphinx_render,
}


# ---------------------------------------------------------
# UNIVERSAL PIPELINE RUNNER
# ---------------------------------------------------------

def run_docstring_pipeline(
    code_input: str,
    style: str = "google",
) -> Dict:
    """
    Runs the complete universal
    docstring generation pipeline.
    """

    # =====================================================
    # VALIDATE STYLE
    # =====================================================

    if style not in RENDERERS:

        raise ValueError(
            (
                f"Unsupported style: {style}\n"
                f"Available styles: "
                f"{list(RENDERERS.keys())}"
            )
        )

    # =====================================================
    # STEP 1 â€” AST EXTRACTION
    # =====================================================

    print("=" * 60)
    print("STEP 1 â€” Extracting function info")
    print("=" * 60)

    function_info = extract_function_info(
        code_input
    )

    print(function_info)

    # =====================================================
    # STEP 2 â€” PROMPT BUILDING
    # =====================================================

    print("\n" + "=" * 60)
    print("STEP 2 â€” Building prompt")
    print("=" * 60)

    prompt = build_prompt(
        function_info
    )

    print(prompt)

    # =====================================================
    # STEP 3 â€” GENERATION
    # =====================================================

    print("\n" + "=" * 60)
    print("STEP 3 â€” Generating docstring")
    print("=" * 60)

    raw_output = generate_docstring(
        prompt
    )

    print(raw_output)

    # =====================================================
    # STEP 4 â€” CLEANING
    # =====================================================

    print("\n" + "=" * 60)
    print("STEP 4 â€” Cleaning docstring")
    print("=" * 60)

    cleaned_docstring = clean_docstring(
        raw_output
    )

    print(cleaned_docstring)

    # =====================================================
    # STEP 5 â€” PARSING
    # =====================================================

    print("\n" + "=" * 60)
    print("STEP 5 â€” Parsing Google docstring")
    print("=" * 60)

    parsed_docstring = (
        parse_google_docstring(
            cleaned_docstring
        )
    )

    print(parsed_docstring)

    # =====================================================
    # STEP 6 â€” NORMALIZATION
    # =====================================================

    print("\n" + "=" * 60)
    print("STEP 6 â€” Normalizing schema")
    print("=" * 60)

    normalized_schema = (
        normalize_docstring_data(
            parsed_docstring=(
                parsed_docstring
            ),
            function_info=(
                function_info
            ),
        )
    )

    print(normalized_schema)

    # =====================================================
    # STEP 7 â€” SEMANTIC VALIDATION
    # =====================================================

    print("\n" + "=" * 60)
    print("STEP 7 â€” Semantic validation")
    print("=" * 60)

    semantic_validation = (
        validate_schema_semantics(
            function_info=(
                function_info
            ),
            schema=(
                normalized_schema
            ),
        )
    )

    print(semantic_validation)

    if not semantic_validation[
        "is_valid"
    ]:

        raise ValueError(
            (
                "Semantic validation failed:\n"
                + "\n".join(
                    semantic_validation[
                        "errors"
                    ]
                )
            )
        )

    # =====================================================
    # STEP 8 â€” STYLE RENDERING
    # =====================================================

    print("\n" + "=" * 60)
    print(
        f"STEP 8 â€” Rendering {style} docstring"
    )
    print("=" * 60)

    renderer = RENDERERS[
        style
    ]

    rendered_docstring = renderer(
        normalized_schema
    )

    print(rendered_docstring)

    # =====================================================
    # STEP 9 â€” STYLE VALIDATION
    # =====================================================

    print("\n" + "=" * 60)
    print(
        f"STEP 9 â€” Validating {style} render"
    )
    print("=" * 60)

    style_validation = {
        "errors": [],
        "is_valid": True,
    }

    if style in STYLE_VALIDATORS:

        validator = STYLE_VALIDATORS[
            style
        ]

        style_validation = validator(
            rendered_docstring
        )

        print(style_validation)

        if not style_validation[
            "is_valid"
        ]:

            raise ValueError(
                (
                    f"{style} render validation "
                    "failed:\n"
                    + "\n".join(
                        style_validation[
                            "errors"
                        ]
                    )
                )
            )

    else:

        print(
            (
                "No style validator registered "
                f'for "{style}".'
            )
        )

    # =====================================================
    # STEP 10 â€” INSERTION
    # =====================================================

    print("\n" + "=" * 60)
    print("STEP 10 â€” Inserting docstring")
    print("=" * 60)

    final_code = insert_docstring(
        function_code=(
            function_info[
                "function_code"
            ]
        ),
        docstring=(
            rendered_docstring
        ),
    )

    print(final_code)

    # =====================================================
    # STEP 11 â€” FINAL AST VALIDATION
    # =====================================================

    print("\n" + "=" * 60)
    print("STEP 11 â€” Final AST validation")
    print("=" * 60)

    try:

        ast.parse(final_code)

        print(
            "Final AST validation passed."
        )

    except SyntaxError as error:

        raise ValueError(
            "Invalid generated code:\n"
            f"{error}"
        )

    # =====================================================
    # SUCCESS
    # =====================================================

    print("\nPipeline completed successfully.")

    return {
        "style": style,
        "function_info": function_info,
        "parsed_docstring": parsed_docstring,
        "normalized_schema": normalized_schema,
        "semantic_validation": (
            semantic_validation
        ),
        "style_validation": (
            style_validation
        ),
        "rendered_docstring": (
            rendered_docstring
        ),
        "final_code": final_code,
    }


# =========================================================
# TEST
# =========================================================

sample_function = """
def add(
    a: int,
    b: int = 0
) -> int:
    return a + b
"""

result = run_docstring_pipeline(
    code_input=sample_function,
    style="numpy",
)

print("\n" + "=" * 60)
print("FINAL OUTPUT")
print("=" * 60)

print(result["final_code"])

STEP 1 â€” Extracting function info
{'function_name': 'add', 'arguments': [{'name': 'a', 'type': 'int', 'default': None}, {'name': 'b', 'type': 'int', 'default': '0'}], 'return_type': 'int', 'has_return_value': True, 'function_code': 'def add(\n    a: int,\n    b: int = 0\n) -> int:\n    return a + b', 'has_docstring': False}

STEP 2 â€” Building prompt
You are an expert Python software engineer.

Generate a canonical Google-style Python docstring.

Requirements:
- Return only the docstring
- Use triple double quotes
- Include Args: if arguments exist
- Include Returns: if the function returns a value
- Use 4-space indentation under sections
- Do not invent arguments

Function:

REQUIRED SECTIONS:
- Args:
- Returns:

CANONICAL FORMAT:
"""
Short description.

Args:
    x (int): Description.

Returns:
    int: Description.
"""

FUNCTION:
def add(
    a: int,
    b: int = 0
) -> int:
    return a + b

STEP 3 â€” Generating docstring
```python
"""
Add two integers.

Args:
    a (int): The 

Final Architecture:
â†’ AST Extraction
â†’ Prompt Building
â†’ Google-style generation
â†’ Cleaning
â†’ Google parsing
â†’ Canonical schema normalization
â†’ Semantic validation
â†’ Rendering a Google / NumPy / Sphinx
â†’ Style validation
â†’ Docstring insertion
â†’ Final AST validation

In [21]:
# =========================================================
# Gradio UI â€” Docstring Generator
# =========================================================

import gradio as gr


# ---------------------------------------------------------
# UI CONSTANTS
# ---------------------------------------------------------

STYLE_OPTIONS = [
    "Google",
    "NumPy",
    "Sphinx",
]


DEFAULT_CODE = '''
def greet(name: str) -> str:
    return f"Hello {name}"
'''


EXAMPLES = [
    [
        '''
def add(a: int, b: int = 0) -> int:
    return a + b
''',
        "Google",
    ],
    [
        '''
def multiply(x: float, y: float) -> float:
    return x * y
''',
        "NumPy",
    ],
]


# ---------------------------------------------------------
# STYLE NORMALIZATION
# ---------------------------------------------------------

def normalize_style_choice(
    style_choice: str,
) -> str:
    """
    Normalizes the UI style choice into the
    internal pipeline style key.
    """

    if not style_choice:

        return "google"

    return style_choice.lower().strip()


# ---------------------------------------------------------
# ERROR FORMATTING
# ---------------------------------------------------------

def format_ui_error(
    error: Exception,
) -> str:
    """
    Formats UI-safe error messages.
    """

    return (
        "âŒ Error:\n\n"
        f"{str(error)}"
    )


# ---------------------------------------------------------
# PIPELINE CALLBACK
# ---------------------------------------------------------

def run_docstring_generation_ui(
    code_input: str,
    style_choice: str,
):
    """
    Gradio callback that runs the universal
    docstring pipeline and returns the rendered
    docstring for the selected style.
    """

    try:

        style = normalize_style_choice(
            style_choice
        )

        pipeline_result = run_docstring_pipeline(
            code_input=code_input,
            style=style,
        )

        rendered_docstring = (
            pipeline_result[
                "rendered_docstring"
            ]
        )

        return (
            rendered_docstring,
            pipeline_result,
        )

    except Exception as error:

        return (
            format_ui_error(error),
            None,
        )


# ---------------------------------------------------------
# THEME
# ---------------------------------------------------------

theme = gr.themes.Soft(
    primary_hue="violet",
    secondary_hue="blue",
    neutral_hue="slate",
)


# ---------------------------------------------------------
# UI
# ---------------------------------------------------------

with gr.Blocks(
    theme=theme,
    title="Docstring Generator ðŸš€",
    css="""
    .gradio-container {
        background: linear-gradient(to bottom right, #0f172a, #1e1b4b);
    }
    """
) as demo:

    gr.Markdown(
        """
# ðŸš€ Docstring Generator

### Turn Python functions into clean docstrings âœ¨
"""
    )

    pipeline_state = gr.State(
        value=None
    )

    with gr.Row():

        with gr.Column():

            code_input = gr.Code(
                label="ðŸ Python Function",
                language="python",
                lines=18,
                value=DEFAULT_CODE,
            )

            style_choice = gr.Dropdown(
                choices=STYLE_OPTIONS,
                value="Google",
                label="ðŸ§  Style",
            )

            generate_button = gr.Button(
                "âœ¨ Generate Docstring",
                variant="primary",
            )

        with gr.Column():

            output_box = gr.Textbox(
                label="ðŸ“„ Generated Docstring",
                lines=18,
                show_copy_button=True,
            )

    generate_button.click(
        fn=run_docstring_generation_ui,
        inputs=[
            code_input,
            style_choice,
        ],
        outputs=[
            output_box,
            pipeline_state,
        ],
    )

    gr.Examples(
        examples=EXAMPLES,
        inputs=[
            code_input,
            style_choice,
        ],
    )


demo.launch(
    debug=False
)

/tmp/ipykernel_2153/112526775.py:138: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_2153/112526775.py:138: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dfbdb884e1f31cc25d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
